#### GROUP 17: Shuvojyoti Singha, Artjom Smorgulenko, Kaan Özkiliç, Samuel Pasierb

# Exploring Unsupervised Learning with the k-Means Algorithm

## Dataset Background
> This dataset contains the answers to 60 questions to determine the personality of a person. The possible personalities are: 
*  ESTJ - The Supervisor
*  ENTJ - The Commander 
*  ESFJ - The Provider 
*  ENFJ - The Giver
*  ISTJ - The Inspector
*  ISFJ - The Nurturer 
*  INTJ - The Mastermind
*  INFJ - The Counselor
*  ESTP - The Doer
*  ESFP - The Performer
*  ENTP - The Visionary
*  ENFP - The Champion 
*  ISTP - The Craftsman
*  ISFP - The Composer 
*  INTP - The Thinker 
*  INFP - The Idealist 
> The answers range from -3 to 3 depending on much the person agrees with the statement. -3 means full disagreement and 3 means full agreement. 

## Imports


In [26]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from matplotlib import pyplot as plt
from enum import Enum

## Constants & Variables


In [27]:
class Personality(Enum):
    ESTJ = 0
    ENTJ = 1
    ESFJ = 2
    ENFJ = 3
    ISTJ = 4
    ISFJ = 5
    INTJ = 6
    INFJ = 7
    ESTP = 8
    ESFP = 9
    ENTP = 10
    ENFP = 11
    ISTP = 12
    ISFP = 13
    INTP = 14
    INFP = 15
    

class Question:
    index: int
    text: str
    
    def __init__(self, index, text) -> None:
        self.index = index
        self.text = text
        

questions: list[Question] = []

## Loading dataset
This dataset is downloaded from Kaggle. It includeds 62 columns, 1 of them being an ID and 1 the Personality, the rest are questions. There was � in the .csv file which had to be replaced with ' manually.

[The link to the dataset on Kaggle.](https://www.kaggle.com/datasets/anshulmehtakaggl/60k-responses-of-16-personalities-test-mbt)

In [28]:
df: pd.DataFrame = pd.read_csv('personality-test-dataset.csv')
df.head()

,Response Id,You regularly make new friends.,You spend a lot of your free time exploring various random topics that pique your interest,Seeing other people cry can easily make you feel like you want to cry too,You often make a backup plan for a backup plan.,"You usually stay calm, even under a lot of pressure","At social events, you rarely try to introduce yourself to new people and mostly talk to the ones you already know",You prefer to completely finish one project before starting another.,You are very sentimental.,You like to use organizing tools like schedules and lists.,...,You believe that pondering abstract philosophical questions is a waste of time.,"You feel more drawn to places with busy, bustling atmospheres than quiet, intimate places.",You know at first glance how someone is feeling.,You often feel overwhelmed.,You complete things methodically without skipping over any steps.,You are very intrigued by things labeled as controversial.,You would pass along a good opportunity if you thought someone else needed it more.,You struggle with deadlines.,You feel confident that things will work out for you.,Personality
0,0,0,0,0,0,0,1,1,0,0,...,0,0,0,-1,0,0,0,0,0,ENFP
1,1,0,0,-2,-3,-1,2,-2,0,3,...,0,-2,0,2,0,-1,-1,-1,3,ISFP
2,2,0,0,2,0,-1,2,0,0,1,...,0,2,0,2,-1,0,1,2,1,INFJ
3,3,0,-1,3,-1,0,0,-2,0,-2,...,0,0,-1,-1,0,1,0,-2,-1,ISTP
4,4,0,0,-1,0,2,-1,-2,0,1,...,0,1,0,2,0,1,-1,2,-1,ENFJ


## Data Cleaning
The missing data is handled and questions are replaced with "Question x" to make the dateset look nicer.

- The missing values are filled with 0.
- ID column is dropped because it is unnecesarry.
- Personiality column (target value) is dropped because it is not needed for clustering
- Questions are replaced with "Question x"

In [ ]:
df.fillna(0, inplace=True)
print(df.isnull().sum())

Response Id                                                                                   0
You regularly make new friends.                                                               0
You spend a lot of your free time exploring various random topics that pique your interest    0
Seeing other people cry can easily make you feel like you want to cry too                     0
You often make a backup plan for a backup plan.                                               0
                                                                                             ..
You are very intrigued by things labeled as controversial.                                    0
You would pass along a good opportunity if you thought someone else needed it more.           0
You struggle with deadlines.                                                                  0
You feel confident that things will work out for you.                                         0
Personality                             

In [30]:
if "Response Id" in df.columns: df = df.drop(["Response Id"], axis=1)
if "Personality" in df.columns: df = df.drop(["Personality"], axis=1)

print(df.columns)

Index(['You regularly make new friends.',
       'You spend a lot of your free time exploring various random topics that pique your interest',
       'Seeing other people cry can easily make you feel like you want to cry too',
       'You often make a backup plan for a backup plan.',
       'You usually stay calm, even under a lot of pressure',
       'At social events, you rarely try to introduce yourself to new people and mostly talk to the ones you already know',
       'You prefer to completely finish one project before starting another.',
       'You are very sentimental.',
       'You like to use organizing tools like schedules and lists.',
       'Even a small mistake can cause you to doubt your overall abilities and knowledge.',
       'You feel comfortable just walking up to someone you find interesting and striking up a conversation.',
       'You are not too interested in discussing various interpretations and analyses of creative works.',
       'You are more inclined to fo

In [31]:
questions = [Question(i + 1, q) for i, q in enumerate(df.columns)]
df.columns = [f"Questions {q.index}" for q in questions]

In [32]:
df.head()

,Questions 1,Questions 2,Questions 3,Questions 4,Questions 5,Questions 6,Questions 7,Questions 8,Questions 9,Questions 10,...,Questions 51,Questions 52,Questions 53,Questions 54,Questions 55,Questions 56,Questions 57,Questions 58,Questions 59,Questions 60
0,0,0,0,0,0,1,1,0,0,0,...,0,0,0,0,-1,0,0,0,0,0
1,0,0,-2,-3,-1,2,-2,0,3,0,...,0,0,-2,0,2,0,-1,-1,-1,3
2,0,0,2,0,-1,2,0,0,1,0,...,0,0,2,0,2,-1,0,1,2,1
3,0,-1,3,-1,0,0,-2,0,-2,0,...,0,0,0,-1,-1,0,1,0,-2,-1
4,0,0,-1,0,2,-1,-2,0,1,0,...,0,0,1,0,2,0,1,-1,2,-1
